In [ ]:
import os
import shutil
import time
from glob import glob
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm

In [ ]:
def change_file_time(file, t):
    timestamp_str = str(t)
    parsed_time = time.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    ts = time.mktime(parsed_time)
    os.utime(file, (ts, ts))

In [ ]:
df = pd.read_csv('wildlife-insights_e28f789b-3c3b-4ad0-9879-e81522823e6a_project-2007160_data/images_2007160.csv')

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
len(df['deployment_id'].unique())

In [ ]:
df = df.sort_values(by='timestamp')

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
# df['location'].to_csv('images_human.csv', index=False, header=False)

```
cat images_human.csv | gsutil cp -I human_images
```

In [ ]:
df['deployment_id'] = df['deployment_id'].str.replace('/', '-')

In [ ]:
df.head()

In [ ]:
for x in df['deployment_id'].unique():
    Path(f'pilotmt/{x}/person').mkdir(exist_ok=True, parents=True)
    Path(f'pilotmt/{x}/animal').mkdir(exist_ok=True)

In [ ]:
preds = pd.read_csv('../yolov5/runs/detect/pilotmt-images/predictions.csv', names=['image', 'pred', 'conf'])

In [ ]:
df['image'] = df['location'].apply(lambda x: Path(x).name)

In [ ]:
df = df.merge(preds, on='image', how='left')

In [ ]:
for x, y, z, t, p in zip(df['location'], df['deployment_id'], df['image'], df['timestamp'], df['pred']):
    if p not in ['person', 'animal'] or not Path(f'images/{z}').exists():
        continue
    out_path = f'pilotmt/{y}/{p}/{z}'
    shutil.copy2(f'images/{z}', out_path)
    change_file_time(out_path, t)